<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/Custom_FAQ_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Cell 1: Setup for 2_1_Assessment (Custom FAQ Chatbot)

# 1. Install official Google GenAI SDK
!pip install -q -U google-genai

import os
from google.colab import userdata
from google.genai import types, client
from google.genai.errors import ClientError

# 2. Retrieve Gemini API key securely from Colab Secrets
api_key = userdata.get('GEMINI_API_KEY')

# 3. Initialize the GenAI Client
ai = client.Client(api_key=api_key)

# 4. Define candidate models ordered by preference to avoid 404 access errors
MODEL_CANDIDATES = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-2.0-flash",
    "gemini-1.5-flash"
]

def generate_with_fallback(prompt, config=None):
    """
    Iterates through candidate models in order to ensure compatibility and bypass 404 errors.
    """
    last_error = None
    for model_name in MODEL_CANDIDATES:
        try:
            res = ai.models.generate_content(
                model=model_name,
                contents=prompt,
                config=config
            )
            print(f"[Success] Response generated using active model: {model_name}\n")
            return res
        except ClientError as e:
            if e.code == 404 or "NOT_FOUND" in str(e):
                print(f"[Notice] Model '{model_name}' unavailable (404). Trying next candidate...")
                last_error = e
                continue
            raise e
    raise RuntimeError(f"All model candidates failed. Last error: {last_error}")

print("Cell 1 Complete: Gemini Client and generate_with_fallback initialized successfully!")

Cell 1 Complete: Gemini Client and generate_with_fallback initialized successfully!


In [7]:
# Cell 2: Contextual Prompting for 2_1_Assessment (Custom FAQ Chatbot)

# 1. Define rich contextual knowledge base for the university course
course_context = """
You are an official AI Teaching Assistant for "CS-101: Introduction to Artificial Intelligence & Large Language Models" at Tech University.

Course Metadata:
- Course Code: CS-101
- Course Title: Introduction to AI & LLMs
- Duration: 12 Weeks (Semester: Fall 2026)
- Prerequisites: Basic Python Programming
- Syllabus Highlights:
  * Weeks 1-3: Foundations of Machine Learning & Neural Networks
  * Weeks 4-6: Transformer Architectures & Self-Attention
  * Weeks 7-9: Fine-Tuning LLMs (LoRA, QLoRA) & PEFT
  * Weeks 10-12: AI Agent Architectures, RAG, & Deployment
- Grading Policy: Assignments (40%), Midterm Project (30%), Final Assessment (30%)
- Office Hours: Tuesdays & Thursdays, 3:00 PM - 5:00 PM IST

Guidelines:
- Answer student questions accurately using ONLY the provided course metadata above.
- Be helpful, polite, and academic in tone.
"""

# 2. Student FAQ Query
student_query = "What topics are covered in the final three weeks, and what is the grading weight for the midterm project?"

# 3. Construct contextual prompt combining course context and student query
contextual_prompt = f"{course_context}\n\nStudent Question: {student_query}\nAnswer:"

# 4. Configure content generation parameters
config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=900
)

# 5. Execute API call using our resilient fallback handler
print("Executing Contextual Prompting Task...\n")
response = generate_with_fallback(contextual_prompt, config=config)

print("=== CONTEXTUAL PROMPT RESPONSE ===")
print(response.text)

Executing Contextual Prompting Task...

[Success] Response generated using active model: gemini-3.6-flash

=== CONTEXTUAL PROMPT RESPONSE ===
Hello!

In the final three weeks of CS-101 (Weeks 10–12), the topics covered are **AI Agent Architectures, RAG (Retrieval-Augmented Generation), & Deployment**.

Regarding the grading breakdown, the **Midterm Project** carries a weight of **30%** of your total grade.

Please let me know if you have any further questions about the course!


In [10]:
# Cell 3: Format Constraints for 2_1_Assessment (Custom FAQ Chatbot)

# 1. Define prompt requiring strict JSON formatting matching a predefined schema
format_constrained_prompt = """
You are an official AI Teaching Assistant for "CS-101: Introduction to AI".

Course Office Hours Metadata:
- Days: Tuesdays & Thursdays
- Time: 3:00 PM - 5:00 PM IST
- Physical Location: Room 302, Computer Science Building
- Online Zoom Link: https://tu.edu/cs101-zoom
- Contact Email: ta-cs101@university.edu

Student Question: "When and where are office hours held, and how can I email the TA?"

Task: Provide the answer strictly in JSON format matching the schema below.
Do NOT include markdown wrapping (e.g. no ```json blocks), conversational filler, or introductory text. Return valid raw JSON only.

JSON Schema:
{
  "question": "string",
  "office_hours": {
    "days": "string",
    "time": "string",
    "location": "string",
    "zoom_link": "string"
  },
  "contact_email": "string"
}
"""

# 2. Configure low temperature for strict schema adherence
config = types.GenerateContentConfig(
    temperature=0.1,
    max_output_tokens=900
)

# 3. Execute API call using our resilient fallback handler
print("Executing Format Constraints Task...\n")
response = generate_with_fallback(format_constrained_prompt, config=config)

print("=== FORMAT CONSTRAINED (STRICT JSON) RESPONSE ===")
print(response.text)

Executing Format Constraints Task...

[Success] Response generated using active model: gemini-3.6-flash

=== FORMAT CONSTRAINED (STRICT JSON) RESPONSE ===
{
  "question": "When and where are office hours held, and how can I email the TA?",
  "office_hours": {
    "days": "Tuesdays & Thursdays",
    "time": "3:00 PM - 5:00 PM IST",
    "location": "Room 302, Computer Science Building",
    "zoom_link": "https://tu.edu/cs101-zoom"
  },
  "contact_email": "ta-cs101@university.edu"
}


In [11]:
# Cell 4: Parameter Control & Iterative Refinement for 2_1_Assessment (Custom FAQ Chatbot)

# 1. Base query regarding course policy
student_query = """
You are an AI FAQ Chatbot for CS-101.
Answer the following student query accurately based on standard academic policy:
"I missed the submission deadline for Assignment 1 because my laptop crashed right before midnight. Can I get a deadline extension or submit it now for partial credit?"
"""

print("Executing Parameter Control Experiment...\n")

# 2. Experiment A: Low Temperature (0.2) - Policy-Strict, Deterministic, Concise
config_low = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=850
)
print("=== EXPERIMENT A: LOW TEMPERATURE (0.2) ===")
response_low = generate_with_fallback(student_query, config=config_low)
print(response_low.text)
print("\n" + "="*60 + "\n")

# 3. Experiment B: High Temperature (0.8) - Empathetic, Conversational, Creative
config_high = types.GenerateContentConfig(
    temperature=0.8,
    max_output_tokens=350
)
print("=== EXPERIMENT B: HIGH TEMPERATURE (0.8) ===")
response_high = generate_with_fallback(student_query, config=config_high)
print(response_high.text)

Executing Parameter Control Experiment...

=== EXPERIMENT A: LOW TEMPERATURE (0.2) ===
[Success] Response generated using active model: gemini-3.6-flash

Hello! I'm sorry to hear that you experienced a laptop crash right before the deadline. Here is the standard policy regarding missed deadlines and technical issues for CS-101:

### 1. Deadline Extensions for Technical Issues
Per standard course policy, **technical issues (such as laptop crashes, Wi-Fi outages, or corrupted files) right before the deadline do not qualify for an unpenalized deadline extension.** 

Students are expected to save and back up their work frequently (e.g., using GitHub, Google Drive, or cloud backups) and submit early to avoid last-minute technical glitches.

### 2. Can You Submit Now for Partial Credit?
**Yes, you should submit your assignment as soon as possible.** 
Most CS-101 courses allow late submissions for partial credit under the following standard Late Policy:
* **Late Penalty:** Submissions made af